# COMP9414 Assignment 1: Search and Planning
***Name:*** Troy Coleman
***zID:*** z5742721

In [159]:
######################## IGNORE #########################
# A global rendering cell (default table allignment was annoying me)
from IPython.display import HTML
HTML("""
<style>
.jp-RenderedMarkdown table, .rendered_html table, .jp-RenderedHTMLCommon table {
    margin-left: 0 !important;
    margin-right: auto !important;
}
</style>
""")
####################### IGNORE #########################

## Part A: Problem Formulation and Search Traces
### A.1 Office graph formulation

![office graph](images/office_graph.png)

#### A1.1
 - **Initial state:** 
     - corridor
 - **Goal test:** 
     - state == lab
 - **Actions:** 
    - Actions(corridor) = {move to mail_room, move to office_a, move to office_b}
    - Actions(lab) = {move to office_a, move to office_b, move to storage}
    - Actions(mail_room) = {move to corridor, move to storage}
    - Actions(office_a) = {move to corridor, move to lab, move to office_b}
    - Actions(office_b) = {move to corridor, move to lab, move to office_a}
    - Actions(storage) = {move to mail_room, move to lab}

    
 - **Transition model:** 
  - **Result(s, move to n) = n**.

 
 - **Path cost: sum of the edge costs** 
    - corridor → office_b → lab = 3 + 3 = 6
    - corridor → office_a → office_b → lab = 2 + 2 + 3 = 7
    - corridor → office_a → lab = 2 + 8 = 10
    - corridor → mail_room → storage → lab = 2 + 2 + 7 = 11
    - corridor → office_b → office_a → lab = 3 + 2 + 8 = 13

#### A.1.2 First three frontier updates
- **DFS (Depth First Search)**:

| Step | Popped (path so far) | Frontier after |
|:-----|:-------|:---------------|
| 1    | (corridor) | (corridor, mail_room) • (corridor, office_a) • (corridor, office_b) |
| 2    | (corridor, mail_room) | (corridor, mail_room, storage) •  (corridor, office_a) • (corridor, office_b) |
| 3    | (corridor, mail_room, storage) | (corridor, mail_room, storage, lab) •  (corridor, office_a) • (corridor, office_b)|

- **BFS (Breadth First Search)**:

| Step | Popped (path so far) | Frontier after |
|:-----|:-------|:---------------|
| 1    | (corridor) | (corridor, mail_room) • (corridor, office_a) • (corridor, office_b) |
| 2    | (corridor, mail_room) | (corridor, office_a) • (corridor, office_b) • (corridor, mail_room, storage) |
| 3    | (corridor, office_a) | (corridor, office_b) • (corridor, mail_room, storage) • (corridor, office_a, lab) • (corridor, office_a, office_b) |

- **UCS (Uniform Cost Search)**:

| Step | Popped (path so far) | Frontier after (sorted by g, path)|
|:-----|:-------|:---------------|
| 1    | g = 0 : (corridor) | (2, (corridor, mail_room)) • (2, (corridor, office_a)) • (3, (corridor, office_b)) |
| 2    | g = 2 : (corridor, mail_room) | (2, (corridor, office_a) • (3, (corridor, office_b)) • (4, (corridor, mail_room, storage) |
| 3    |g = 2 : (corridor, office_a)| (3, (corridor, office_b) • (4, (corridor, mail_room, storage) • (4, corridor, office_a, office_b) • (10, (corridor, office_a, lab)) |


#### A.1.3 Expansion order, A.1.4 Returned path and cost
| Algorithm | Expansion order | Returned path | Returned cost |
|:--|:--|:--|:--|
| DFS | corridor, mail_room, storage, lab | (corridor, mail_room, storage, lab) | 11 |
| BFS | corridor, mail_room, office_a, office_b, storage, lab | (corridor, office_a, lab) | 10 |
| UCS | corridor, mail_room, office_a, office_b, storage, lab | (corridor, office_b, lab) | 6 |


#### A.1.5 Why BFS and UCS differ
BFS orders by depth (in number of edges) whereas UCS weights a cumulative cost of all the edges. Because the two routes tie on BFS's only criterion (both two edges deep, 2+8 = 10 via office_a and 3+3 = 6 via office_b). BFS queue ordering places the 10 cost path earlier in its queue. UCS instead orders by cumulative cost. Since 6 is lower, that is the path UCS returns. 

### A.2 What is the difference between a State Space and a Search Tree?
State space describes the world, encompassing all entities and their valid states. In the example above the state space is the graph structure. Search trees are constructed algorithmically by navigating the state space. Here, the search trees are constructed using BFS, DFS and UCS search algorithms. Typically, algorithms that construct search trees need to retain a history of all visited nodes in the state space to prevent infinite search tree expansion, while state spaces are finite. In my BFS trace (above) the step-3 frontier contained both (corridor, office_b) and (corridor, office_a, office_b): the single state office_b appears as two search-tree nodes because each tree node is a path, and office_b is reachable by two routes with different costs (3 and 4)."

## Part B: Graph search implementation and comparison

Running all four algorithms on office.json, graph2.json, graph3.json.

In [318]:
import json
import math
import heapq
from collections import deque


def load_problem(path):
    with open(path) as f:
        return json.load(f)


def build_adjacency(problem):
    adj = {node["id"]: {} for node in problem["nodes"]}
    for e in problem["edges"]:
        adj[e["u"]][e["v"]] = e["cost"]
        adj[e["v"]][e["u"]] = e["cost"]
    return adj


def empty_result():
    return {"status": "error", "path": [], "plan": [], "cost": None,
            "expanded_order": [], "expanded_count": 0, "generated_count": 0,
            "frontier_peak": 0, "message": None}


def path_cost(adj, p):
    return sum(adj[p[i]][p[i + 1]] for i in range(len(p) - 1))


def run_search(problem, search_fn, **kwargs):
    solved = empty_result()
    try:
        adj = build_adjacency(problem)
        solution = search_fn(adj, problem["start"], problem["goal"], **kwargs)
        if solution is None:
            solved["status"] = "not_found"
            return solved
        p = solution["path"]
        solved["status"] = "found"
        solved["path"] = list(p)
        solved["cost"] = solution["cost"] if solution["cost"] is not None else path_cost(adj, p)
        solved["expanded_order"] = solution["expanded_order"]
        solved["expanded_count"] = len(solution["expanded_order"])
        solved["generated_count"] = solution["generated_count"]
        solved["frontier_peak"] = solution["largest_frontier"]
    except Exception as e:
        solved["message"] = str(e)
    return solved


def print_result(name, result):
    print(f"--- {name} ---")
    for field in ["status", "path", "cost", "expanded_count",
                  "generated_count", "frontier_peak", "expanded_order"]:
        print(f"  {field}: {result[field]}")
    print()

In [320]:
def dfs(graph, start, goal):
    frontier = [(start,)]
    largest_frontier = 1
    explored = set()
    expanded_order = []
    generated_count = 0

    while frontier:
        path = frontier.pop()
        node = path[-1]
        if node in explored:
            continue
        explored.add(node)
        expanded_order.append(node)
        if node == goal:
            return {"path": path, "cost": None,
                    "expanded_order": expanded_order,
                    "largest_frontier": largest_frontier,
                    "generated_count": generated_count}
        for n in sorted(graph[node], reverse=True):
            generated_count += 1
            if n not in explored:
                frontier.append(path + (n,))
                largest_frontier = max(largest_frontier, len(frontier))
    return None

In [321]:
def bfs(graph, start, goal):
    frontier = deque([(start,)])
    # TODO: initialise largest_frontier, expanded_order, generated_count  (Cell 2, lines 3–5)
    explored = set()

    while frontier:
        path = frontier.popleft()
        node = path[-1]
        if node in explored:
            continue
        explored.add(node)
        # TODO: expanded_order line  (Cell 2)
        if node == goal:
            pass  # TODO: return the same dict as Cell 2, "cost": None
        for n in sorted(graph[node]):
            # TODO: generated_count line  (Cell 2)
            if n not in explored:
                frontier.append(path + (n,))
                # TODO: largest_frontier max-check  (Cell 2)
    return None

In [322]:
def ucs(graph, start, goal):
    counter = 0
    frontier = [(0, (start,), 0)]
    # TODO: initialise largest_frontier, expanded_order, generated_count
    explored = set()

    while frontier:
        g, path, _ = heapq.heappop(frontier)
        node = path[-1]
        if node in explored:
            continue        # stale entry: no expanded_order append here
        explored.add(node)
        # TODO: expanded_order line
        if node == goal:
            pass  # TODO: return the Cell 2 dict, but with "cost": g
        for n in sorted(graph[node]):
            # TODO: generated_count line  (note: 'counter' below is a different variable — keep both)
            if n not in explored:
                counter += 1
                heapq.heappush(frontier, (g + graph[node][n], path + (n,), counter))
                # TODO: largest_frontier max-check
    return None

In [323]:
def astar(graph, start, goal, coords):
    def h(n):
        return math.dist(coords[n], coords[goal])

    counter = 0
    frontier = [(h(start), (start,), 0, 0)]      # (f, path, counter, g)
    # TODO: same three initialisations
    explored = set()

    while frontier:
        f, path, _, g = heapq.heappop(frontier)
        node = path[-1]
        if node in explored:
            continue
        explored.add(node)
        # TODO: expanded_order line
        if node == goal:
            pass  # TODO: Cell 2 dict with "cost": g
        for n in sorted(graph[node]):
            # TODO: generated_count line
            if n not in explored:
                counter += 1
                new_g = g + graph[node][n]
                heapq.heappush(frontier, (new_g + h(n), path + (n,), counter, new_g))
                # TODO: largest_frontier max-check
    return None

In [324]:
def solve_graph(problem: dict, algorithm: str) -> dict:
    if algorithm == "astar":
        coords = {n["id"]: (n["x"], n["y"]) for n in problem["nodes"]}
        return run_search(problem, astar, coords=coords)
    searchers = {"dfs": dfs, "bfs": bfs, "ucs": ucs}
    if algorithm not in searchers:
        r = empty_result()
        r["message"] = f"unknown algorithm: {algorithm}"
        return r
    return run_search(problem, searchers[algorithm])

In [325]:
graph_files = {
    "office": "data/graphs/office.json",
    "graph2": "data/graphs/graph2.json",
    "graph3": "data/graphs/graph3.json",
}

for gname, gpath in graph_files.items():
    problem = load_problem(gpath)
    for algo in ["dfs", "bfs", "ucs", "astar"]:
        result = solve_graph(problem, algo)
        print_result(f"{gname} / {algo}", result)

NameError: name 'bfs' is not defined

In [326]:
import matplotlib.pyplot as plt


ModuleNotFoundError: No module named 'matplotlib'

## Part C: A* heuristic behaviour analysis

Comparing UCS and A* on two graphs; identify where the heuristic
changes a frontier choice; discuss the limitation of coordinate
distance as a guide.

In [327]:
# TODO: run UCS and A* on chosen graphs, compare metrics, build table.

## Part D: STRIPS modelling and forward planning

### D.1 Plan diagnosis (lab_via_office_b)
1. First inapplicable action: ...
2. Missing precondition(s) / false fact(s): ...
3. Why inserting unlock_door alone isn't enough: ...
4. Repaired plan: ...
5. Where unlocked(lab_door) first becomes true: ...
6. How the final state satisfies at(parcel, lab): ...

In [285]:
def solve_planning(problem: dict) -> dict:
    """
    Forward BFS STRIPS planner.
    Returns a dict with the Table 2 fields (plan instead of path):
      status, plan, cost, expanded_order,
      expanded_count, generated_count, frontier_peak, (message)
    Use a canonical state id (e.g. sorted tuple of facts) for
    visited-tracking and expanded_order.
    """
    # TODO: implement forward BFS over fact-set states here.
    raise NotImplementedError

In [286]:
planning_files = {
    "canonical_delivery": "data/planning/canonical_delivery.json",
    "unlocked_lab": "data/planning/unlocked_lab.json",
}

for pname, ppath in planning_files.items():
    problem = load_problem(ppath)
    result = solve_planning(problem)
    print_result(pname, result)

NotImplementedError: 

## Reusable evaluation cell

Loads all supplied examples, runs every required algorithm, prints
a readable table, and writes the summary to zID_results.json.

In [198]:
summary = {}

# Graph problems: all four algorithms
for gname, gpath in graph_files.items():
    problem = load_problem(gpath)
    summary[gname] = {}
    for algo in ["dfs", "bfs", "ucs", "astar"]:
        summary[gname][algo] = solve_graph(problem, algo)

# Planning problems
for pname, ppath in planning_files.items():
    problem = load_problem(ppath)
    summary[pname] = solve_planning(problem)

# TODO: print summary as a readable table here.

with open("z5742721_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Wrote z5742721_results.json")

DFS Problem {'nodes': [{'id': 'corridor', 'x': 0.0, 'y': 0.0}, {'id': 'mail_room', 'x': -1.0, 'y': 0.0}, {'id': 'office_a', 'x': 0.0, 'y': 1.0}, {'id': 'office_b', 'x': 1.0, 'y': 0.0}, {'id': 'storage', 'x': -2.0, 'y': 0.0}, {'id': 'lab', 'x': 2.0, 'y': 0.0}], 'edges': [{'u': 'corridor', 'v': 'mail_room', 'cost': 2.0}, {'u': 'corridor', 'v': 'office_a', 'cost': 2.0}, {'u': 'corridor', 'v': 'office_b', 'cost': 3.0}, {'u': 'mail_room', 'v': 'storage', 'cost': 2.0}, {'u': 'storage', 'v': 'lab', 'cost': 7.0}, {'u': 'office_a', 'v': 'lab', 'cost': 8.0}, {'u': 'office_b', 'v': 'lab', 'cost': 3.0}, {'u': 'office_a', 'v': 'office_b', 'cost': 2.0}], 'start': 'corridor', 'goal': 'lab', 'metadata': {'source': 'assignment1_hand_specified', 'description': 'Hand-specified office graph from Part A.'}}
BFS Problem {'nodes': [{'id': 'corridor', 'x': 0.0, 'y': 0.0}, {'id': 'mail_room', 'x': -1.0, 'y': 0.0}, {'id': 'office_a', 'x': 0.0, 'y': 1.0}, {'id': 'office_b', 'x': 1.0, 'y': 0.0}, {'id': 'storage',

NotImplementedError: 